# Baseline CNN on CIFAR-10

Run this notebook in Google Colab with `Runtime > Change runtime type > GPU`.

The notebook trains the baseline CNN, saves a CSV log, saves a checkpoint, evaluates on the test set, and writes a compact JSON summary.

In [ ]:
# If you opened this notebook outside the cloned repository, clone it first.
# Change BRANCH if you want to run a different branch.
import os
from pathlib import Path

REPO_URL = "https://github.com/iwadas/GGSN-project.git"
BRANCH = "dataset_preparation"
REPO_DIR = Path("/content/GGSN-project")

if not Path("pyproject.toml").exists():
    os.chdir("/content")
    if not REPO_DIR.exists():
        !git clone -b {BRANCH} {REPO_URL}
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

## Install Dependencies

Colab usually already provides a CUDA-enabled PyTorch build. This cell installs the project extras without replacing the runtime more than necessary.

In [ ]:
%pip install -q uv
!uv pip install --system -q optuna numpy pandas matplotlib pyyaml tqdm

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    batch_size: int = 64
    num_workers: int = 2
    epochs: int = 5
    learning_rate: float = 1e-3
    num_layers: int = 3
    base_filters: int = 32
    dropout: float = 0.2

config = Config()
config

## Train Baseline CNN

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from torch import nn

from data.dataloader import get_cifar10_dataloaders
from evaluation.metrics import count_parameters, evaluate, measure_inference_latency
from models.baseline_cnn import build_baseline_cnn
from training.trainer import fit

Path("results").mkdir(exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)

train_loader, validation_loader, test_loader = get_cifar10_dataloaders(
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    download=True,
)

model = build_baseline_cnn(
    num_layers=config.num_layers,
    base_filters=config.base_filters,
    dropout=config.dropout,
)
parameter_count = count_parameters(model)
print("Parameters:", parameter_count)

optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
criterion = nn.CrossEntropyLoss()

training_result = fit(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=config.epochs,
    checkpoint_path="checkpoints/baseline_cnn.pt",
    log_csv_path="results/baseline_training_log.csv",
)

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = torch.load("checkpoints/baseline_cnn.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

test_result = evaluate(model, test_loader, criterion, device)
latency_ms = measure_inference_latency(model, device=device)

summary = {
    "epochs": config.epochs,
    "batch_size": config.batch_size,
    "learning_rate": config.learning_rate,
    "num_layers": config.num_layers,
    "base_filters": config.base_filters,
    "dropout": config.dropout,
    "parameters": parameter_count,
    "best_validation_accuracy": training_result.best_validation_accuracy,
    "test_loss": test_result.loss,
    "test_accuracy": test_result.accuracy,
    "latency_ms": latency_ms,
    "device": device,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

with open("results/baseline_summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

summary

## Show Training Curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = pd.read_csv("results/baseline_training_log.csv")
display(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["validation_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["epoch"], history["train_accuracy"], label="train")
axes[1].plot(history["epoch"], history["validation_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

fig.tight_layout()
fig.savefig("results/baseline_training_curves.png", dpi=150)
plt.show()

## Files To Keep

Useful generated artifacts:

- `results/baseline_training_log.csv`
- `results/baseline_summary.json`
- `results/baseline_training_curves.png`

The checkpoint `checkpoints/baseline_cnn.pt` is useful for resuming/evaluation, but usually should not be committed if it is large.

In [ ]:
!ls -lh results checkpoints